In [ ]:
import numpy as np
import os
base_folder = '/home/leejs/MeshRIR/S1-M3969_npy'
eval        = False
seq_len     = int(24000 * 0.1)
fs          = 24000
wave_max = float('-inf')
wave_min = float('inf')
position_max = np.array([float('-inf'), float('-inf'), float('-inf')])
position_min = np.array([float('inf'), float('inf'), float('inf')])
wave_chunks  = []
positions_rx = []
positions_tx = []
rotations_tx = []

def get_st_idx(audio_data, position_rx, position_tx, fs):
    r = np.linalg.norm(position_rx - position_tx) # rx와 tx 사이 거리
    onset = get_onset(audio_data)
    return onset - int(r * fs / 343) # onset 위치를 기준으로, 거리에 의한 전파지연만큼을 앞당겨서 샘플 채취
    # 일단 논문이 잘 쓰였으니 큰 문제는 아닌 듯 하다. 다만 위험한 요소이긴 함

def get_onset(audio_data):
    x_abs = np.abs(audio_data)
    x_max = x_abs.max()
    indices = np.where(np.abs(audio_data) > 0.1 * x_max)[0]
    onset = indices[0] # 임펄스 크기의 10% 인 피크가 나오는 가장 첫 시점 (유효한 신호가 존재하기 시작하는 시점)
    return onset       

def update_min_max(audio_data, position_rx):
    global wave_max, wave_min, position_max, position_min
    wave_max = max(wave_max, audio_data.max())
    wave_min = min(wave_min, audio_data.min())
    position_max = np.maximum(position_max, position_rx)
    position_min = np.minimum(position_min, position_rx)
    return wave_max, wave_min, position_max, position_min

def load_mesh_rir(base_folder, eval, seq_len, fs=24000):
    """ Load MeshRIR datasets
    """
    down_sample_rate = 48000 // fs

    if eval:
        wave_folder = os.path.join(base_folder, 'test')
    else:
        wave_folder  = base_folder
        #wave_folder = os.path.join(base_folder, 'train')

    filenames = [f for f in os.listdir(wave_folder) if f.endswith('.npy')]
    filenames.sort()

    rx_pos = np.load(os.path.join(base_folder, 'pos_mic.npy'))    # [3969, 3]
    tx_pos = np.load(os.path.join(base_folder, 'pos_src.npy'))[0] #  [1, 3]

    for filename in filenames:
        global wave_chunks, positions_rx, positions_tx
        try:
            file_ind = int(filename.split('_')[1].split('.')[0])
        except:
            continue
        position_rx = rx_pos[file_ind] # [3, ]
        position_tx = tx_pos           # [3, ]
        audio_data = np.load(os.path.join(wave_folder, filename))[0,::down_sample_rate] # [16384, ]
        st_idx = get_st_idx(audio_data, position_rx, position_tx, fs)
        wave_data = audio_data[st_idx:st_idx+seq_len] # 여기서는 길이 2400짜리 소리 데이터가 맞는데
        wave_data = np.fft.rfft(wave_data)            # rfft를 해서 1200개의 complex data로 바꿈
        update_min_max(audio_data, position_rx)
        wave_chunks.append(wave_data)
        positions_rx.append(position_rx)
        positions_tx.append(position_tx)

load_mesh_rir(base_folder, eval, seq_len, fs=24000)

In [17]:
# 목표: wave_max, wave_min, position_max, position_min, wave_chunks, positions_rx, positions_tx 이렇게 7개 만들기
print(wave_max, type(wave_max)) # 스칼라, numpy.float64
print(wave_min, type(wave_min)) # 스칼라, numpy.float64
print(position_max, type(position_max)) # numpy.ndarray, [3, ]
print(position_min, type(position_min)) # numpy.ndarray, [3, ]
print(wave_chunks.__len__(), type(wave_chunks[0][0]))   # 3969개(리시버 개수임) 리스트이고, 한 entity가 (1201, ) shape임. 1201개 값 각각은 numpy.complex128
print(positions_rx[0], type(positions_rx[0][0]))        # 3969개(리시버 개수임) 리스트이고, 한 entity가 (3, )    shape임. 각각은 numpy.float64
print(positions_tx[0], type(positions_tx))              # 3969개(리시버 개수임) 리스트이고, 한 entity가 (3, )    shape임. 각각은 numpy.float64
print(rotations_tx.__len__(), type(rotations_tx))       # 죽은 코드임 (안 구현해도 무방)


1.5737298754140996 <class 'numpy.float64'>
-1.174269150521193 <class 'numpy.float64'>
[0.5 0.5 0.2] <class 'numpy.ndarray'>
[-0.5 -0.5 -0.2] <class 'numpy.ndarray'>
3969 <class 'numpy.complex128'>
[-0.5 -0.5 -0.2] <class 'numpy.float64'>
[2.  1.5 0. ] <class 'list'>
0 <class 'list'>


In [18]:
print(positions_tx)

[array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5, 0. ]), array([2. , 1.5

In [11]:
import torch
import pandas as pd
import librosa
import os
import re
import numpy as np
def _room_prefix(roomnum):
    return f"R{roomnum:02d}"

def _load_room_file(data_dir, roomnum, t_max=0.1, status_ok="Success"):
    csv_path = os.path.join(data_dir, "Generated_HOA_SRIR_data.csv")
    metadata = pd.read_csv(csv_path)
    prefix = _room_prefix(roomnum)
    room_mask = metadata['Name'].astype(str).str.match(rf'^{re.escape(prefix)}[_\-]?')
    if status_ok is not None and 'status' in metadata.columns:
        room_mask &= (metadata['status'] == status_ok)
    room_meta = metadata[room_mask].reset_index(drop=True)
    if len(room_meta) == 0:
        raise ValueError(f"no rows found for room '{prefix}' (status='{status_ok}') in {csv_path}")
    x = room_meta['x_mic'].to_numpy(dtype=np.float32)
    y = room_meta['y_mic'].to_numpy(dtype=np.float32)
    z = room_meta['z_mic'].to_numpy(dtype=np.float32)

    foa_list = []
    fs = None
    n_samples = None
    SN3D_MISSING_GAIN = np.sqrt(3.0)   # SN3D Normalization이 적용되지 않은 채로 생성된 음원임
    RAW_TO_WXYZ = [0, 3, 1, 2]         # ACN Format
    for name in room_meta['Name']:
        wav_path = os.path.join(data_dir, name)
        audio, file_fs = librosa.load(wav_path, sr=None, mono=False)
        if audio.ndim != 2 or audio.shape[0] != 4:
            raise ValueError(f"{wav_path}: expected 4-channel FOA audio, got shape {audio.shape}")
        if fs is None:
            fs = file_fs
            n_samples = min(int(round(t_max * fs)), audio.shape[1])
        elif file_fs != fs:
            raise ValueError(f"sample rate mismatch in {wav_path}: {file_fs} != {fs}")
        audio = audio.copy()
        audio[1:4, :] /= SN3D_MISSING_GAIN
        audio = audio[RAW_TO_WXYZ, :]
        foa_list.append(audio[:, :n_samples].T)  # (n_samples, 4)

    foa = np.stack(foa_list, axis=0).astype(np.float32)
    # foa maxabs 정규화로 0 ~ 1 사이로 늘림
    target_peak = 0.9
    foa_maxabs  = float(np.max(np.abs(foa))) + 1e-12
    foa_scale   = target_peak / foa_maxabs
    foa = foa * foa_scale
    # 
    t = (np.arange(foa.shape[1]) / fs).astype(np.float32)
    omega_center = np.array([
        x.mean(),
        y.mean(),
        z.mean(),
    ], dtype=np.float32)
    omega_size = 1.0
    # x, y, z는 raw 좌표값이 들어가고, t는 0 ~ 0.1 사이를 800등분한 시간축 좌표가 들어감
    return dict(x=x, y=y, z=z, foa=foa, t=t, omega_center=omega_center,
               omega_size=omega_size, t_max=float(t[-1]) if len(t) else t_max,
               meta=room_meta) # 파일들은 정상적으로 잘 슬라이싱되어 들어가고 있음

def compute_bounds(omega_center, omega_size, t_max):
    half = omega_size / 2.0
    lb = torch.tensor([1.1 * (omega_center[0] - half), 1.1 * (omega_center[1] - half),
                       1.1 * (omega_center[2] - half), 0.0], dtype=torch.float32)
    ub = torch.tensor([1.1 * (omega_center[0] + half), 1.1 * (omega_center[1] + half),
                       1.1 * (omega_center[2] + half), t_max], dtype=torch.float32)
    return lb.numpy(), ub.numpy()

room = _load_room_file("/home/leejs/HARP/Generated_HOA_IRs(azimuth_fixed)", 0)


In [25]:
lb, ub = compute_bounds(room['omega_center'], room['omega_size'], room['t_max'])
print(lb, ub)

[1.3673317 3.0690763 0.7321142 0.       ] [2.467332  4.1690764 1.8321142 0.099875 ]


In [24]:
center       = np.array([0.5, 0.5, 0.5])
center       = center + 1
print(center)
a = np.repeat(1, 3)
b = np.repeat(2, 3)
c = np.repeat(3, 3)
d = np.stack([a,b,c], axis = -1)
print(type(d))
e = list(d)
print(e, type(e))

[1.5 1.5 1.5]
<class 'numpy.ndarray'>
[array([1, 2, 3]), array([1, 2, 3]), array([1, 2, 3])] <class 'list'>


In [3]:
import numpy as np
position_tx_pde = np.stack([1,2,3], axis = -1)
print(position_tx_pde)

[1 2 3]
